# Chapter 7: stage table, loss waterfall and reconcile check

The Kestrel month: 8,700 hL cast, 8,330 hL dispatched. The reconcile card must read zero. Synthetic data only.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = Path("../data")   # run this notebook from the notebooks folder
def load(name, **kw):
    return pd.read_csv(DATA / f"{name}.csv", **kw)

In [ ]:
vol = load("Volume").set_index("Month")
st = load("Stage")
month = "2026-03-01"
s = st[st.Month == month].sort_values("StageOrder")
cast, disp = vol.loc[month, "CastHL"], vol.loc[month, "DispatchedHL"]
print(s[["Stage", "InHL", "OutHL", "LossHL"]].to_string(index=False))
reconcile = cast - s.LossHL.sum() - disp
print("reconcile (must be 0):", reconcile, "| loss", s.LossHL.sum(), "hL =", round(100 * s.LossHL.sum() / cast, 2), "percent")

## The waterfall, cast to dispatch, as a stacked column with an invisible base

In [ ]:
labels = ["Wort cast"] + s.Stage.tolist() + ["Dispatched"]
tops = [cast] + s.InHL.tolist() + [disp]
bases = [0] + s.OutHL.tolist() + [0]
heights = [t - b for t, b in zip(tops, bases)]
colours = ["0.35"] + ["tab:orange"] * 3 + ["0.7"] + ["0.35"]
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(labels, heights, bottom=bases, color=colours)
for i, (h, bt) in enumerate(zip(heights, bases)):
    ax.text(i, bt + h + 60, f"{h:,.0f}", ha="center", fontsize=8)
ax.set_ylim(0, 9500); ax.set_ylabel("hL"); ax.set_title("Beer loss, cast to dispatch, March 2026, hL", loc="left")
plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()

## The Pareto with money at replacement cost

In [ ]:
p = s.sort_values("LossHL", ascending=False).assign(
    share=lambda d: 100 * d.LossHL / d.LossHL.sum(),
    running=lambda d: (100 * d.LossHL.cumsum() / d.LossHL.sum()).round(1),
    usd=lambda d: d.LossHL * d.ReplacementUSDperHL,
    peak=lambda d: d.LossHL * d.PeakUSDperHL)
print(p[["Stage", "LossHL", "running", "usd", "peak"]].to_string(index=False))
print("USD a month", p.usd.sum(), "| a year", p.usd.sum() * 12, "| peak upper bound a month", p.peak.sum())

## Brewhouse yield as a ratio of sums

In [ ]:
brew = load("Brew", parse_dates=["CastDate"])
mar = brew[brew.CastDate.dt.month == 3]
print("one brew B26-0068:", brew.set_index("BrewId").loc["B26-0068", ["ExtractKg", "TheoreticalKg"]].tolist())
print(f"March yield {100 * mar.ExtractKg.sum() / mar.TheoreticalKg.sum():.1f} percent from {len(mar)} brews")
print(f"Year yield {100 * brew.ExtractKg.sum() / brew.TheoreticalKg.sum():.1f} percent from {len(brew)} brews")